# Historical Weather Data by ZIP Code (Open-Meteo) - Databricks SQL Version

This notebook retrieves daily historical weather data for US ZIP codes using the Open-Meteo API. It supports bulk fetching for the past 2 years and incremental updates for recent days. Data is stored at the ZIP code level with basic weather columns in Databricks tables.

> **Note:** This notebook previously used the Meteostat API but has been updated to use Open-Meteo API for better data quality and coverage.

All the weather data from the Open-Meteo API is provided in metric units:

* Temperature values (tavg, tmin, tmax): Celsius (°C)
* Precipitation (prcp): Millimeters (mm)
* Snow depth (snow): Centimeters (cm)
* Wind speed (wspd): Kilometers per hour (km/h)

Example conversion to Fahrenheit  
* `df['tavg_f'] = df['tavg'] * 9/5 + 32`  
* `df['tmin_f'] = df['tmin'] * 9/5 + 32`  
* `df['tmax_f'] = df['tmax'] * 9/5 + 32`

In [0]:
%pip install pgeocode openmeteo-requests requests-cache retry-requests pandas

In [0]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pgeocode
# Import Open-Meteo libraries instead of Meteostat
import openmeteo_requests
import requests_cache
from retry_requests import retry
from pyspark.sql.functions import current_timestamp, lit, col

## CONFIG
Set your parameters here. You can control the ZIP codes, date ranges, and whether to refetch bulk data.

In [0]:
# --- CONFIGURATION FLAGS ---
ZIP_CODES = ["33037", "10001", "60601", "90210", "77002"]  # Example ZIP codes

# API config
COUNTRY = "US"
BULK_YEARS = 2  # How many years back for bulk fetch
N_DAYS = 7  # How many recent days to fetch with incremental API

# Tables
BULK_DATA_TABLE = "cma_users_esrivast.weather_data_5_26_25"  # name of the final weather pipeline table with all data
ZIP_STORE_TBL = "cma_users_esrivast.weather_zip_store_mapping" # name of table with zip code to store number mapping 

# Refetch Flags
ZIP_STORE_REFETCH = False # Set True to force refetch of zip store mapping from enterprise_store_detail
REFETCH_BULK = True  # Set True to force refetch of bulk data from API

## Fetch Zip Code + Store Mapping
- Edit the filters as needed

In [0]:
if ZIP_STORE_REFETCH:
  # adjust filters as needed

  df_zip_store = spark.sql(f'''
    Select  StoreNumber,
            StoreStatusCode,
            LocationTypeName,
            StorePostalCode as ZipCode,
            left(StorePostalCode,5) as Zip5Code,
            StoreCountrySubdivisionCode as State,
            StoreCityName,
            StoreLocationLatitude,
            StoreLocationLongitude,
            StoreOwnershipTypeCode,
            DriveThruTypeCode,
            StoreOpenDate,
            ProposedStoreOpenDate
    from edap_pub_store.enterprise_store_detail
    where 1=1
    and StoreCountryCode = 'USA'
    and StoreStatusCode not in ('dead','develop')
    and StorePostalCode is not null
  ''')

  # overwrite existing table 
  spark.sql(f"DROP TABLE IF EXISTS {ZIP_STORE_TBL}")
  df_zip_store.write.format("delta").mode("overwrite").saveAsTable(ZIP_STORE_TBL)

  print(f"Data written to {ZIP_STORE_TBL} successfully")

else:
  # read from existing table
  print("Fetching zip + store mapping from existing table: ",ZIP_STORE_TBL)
  df_zip_store = spark.table(ZIP_STORE_TBL)

df_zip_store = df_zip_store.toPandas()
display(df_zip_store)

## Convert ZIP Codes to Coordinates

In [0]:
def zip_to_point(zip_codes, country=COUNTRY):
    nomi = pgeocode.Nominatim(country)
    df = nomi.query_postal_code(zip_codes)
    df = df[['postal_code', 'latitude', 'longitude']].rename(columns={'postal_code': 'Zip5Code'})
    df = df.dropna(subset=['latitude', 'longitude'])
    df['Zip5Code'] = df['Zip5Code'].astype(str)
    return df

zip_df = zip_to_point(ZIP_CODES)
display(zip_df)

In [0]:
def add_coordinates_to_df(df_zip_store, country=COUNTRY):
    # Extract unique ZIP codes to minimize API calls
    unique_zip_codes = df_zip_store['Zip5Code'].unique().tolist()
    
    # Get coordinates for unique ZIP codes
    coords_df = zip_to_point(unique_zip_codes, country)
    
    # Create a lookup dictionary for fast mapping
    zip_to_coords = coords_df.set_index('Zip5Code').to_dict('index')
    
    # Add latitude and longitude columns using dictionary mapping
    df_zip_store['latitude'] = df_zip_store['Zip5Code'].map(
        lambda z: zip_to_coords.get(z, {}).get('latitude')
    )
    df_zip_store['longitude'] = df_zip_store['Zip5Code'].map(
        lambda z: zip_to_coords.get(z, {}).get('longitude')
    )
    
    # Return the DataFrame with the new columns
    return df_zip_store

# Apply the function to your DataFrame
df_zip_store = add_coordinates_to_df(df_zip_store)

In [0]:
display(df_zip_store)

In [0]:
display(
  df_zip_store[df_zip_store['Zip5Code'] == '33037']
)

## Helper function to check if a table exists

In [0]:
def table_exists(table_name):
    """Check if a table exists in Databricks"""
    return spark.catalog.tableExists(table_name)

In [0]:
table_exists(BULK_DATA_TABLE)

## Bulk Fetch: Past 2 Years Daily Weather by ZIP Code

In [0]:
unique_zips_df = df_zip_store[['Zip5Code','latitude','longitude']].drop_duplicates(subset=['Zip5Code']).sort_values('Zip5Code').reset_index(drop=True)
display(unique_zips_df)

In [0]:
def fetch_bulk_weather(zip_df, years=2):
    # Setup the Open-Meteo API client with cache and retry
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    # Define start and end dates
    end = datetime.now().date() - timedelta(days=1)
    start = end - timedelta(days=365*years)
    
    # Format dates as strings for the API
    start_str = start.strftime("%Y-%m-%d")
    end_str = end.strftime("%Y-%m-%d")
    
    # Get unique ZIP codes to avoid duplicate API calls
    unique_zips_df = zip_df[['Zip5Code','latitude','longitude']].drop_duplicates(subset=['Zip5Code']).sort_values('Zip5Code').reset_index(drop=True)

    print(f"Fetching weather data for {len(unique_zips_df)} unique ZIP codes (from {len(zip_df)} total records)")
    
    # Define the API parameters
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    # Fetch data for unique ZIP codes
    records = []
    for idx, row in unique_zips_df.iterrows():
        lat, lon = row['latitude'], row['longitude']
        print(f"Fetching for zip code #{idx+1} out of {len(unique_zips_df)}: {row['Zip5Code']}")
        
        # Define parameters for this location
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_str,
            "end_date": end_str,
            "daily": ["temperature_2m_mean", "temperature_2m_min", "temperature_2m_max", 
                      "precipitation_sum", "snowfall_sum", "wind_speed_10m_max"],
            "timezone": "auto"
        }
        
        try:
            # Make the API request
            responses = openmeteo.weather_api(url, params=params)
            response = responses[0]
            
            # Process daily data
            daily = response.Daily()
            
            # Process the Time() values which are returned as Unix timestamp strings
            daily_time_values = daily.Time()
            
            # Convert Unix timestamps (in seconds) to datetime objects
            # pd.to_datetime() automatically converts integers or timestamp strings to datetime
            # with unit='s' parameter to indicate the values are in seconds
            daily_data = {
                'time': pd.to_datetime(daily_time_values, unit='s'),  # Convert Unix timestamps to datetime
                'tavg': daily.Variables(0).ValuesAsNumpy(),  # temperature_2m_mean
                'tmin': daily.Variables(1).ValuesAsNumpy(),  # temperature_2m_min
                'tmax': daily.Variables(2).ValuesAsNumpy(),  # temperature_2m_max
                'prcp': daily.Variables(3).ValuesAsNumpy(),  # precipitation_sum
                'snow': daily.Variables(4).ValuesAsNumpy(),  # snowfall_sum 
                'wspd': daily.Variables(5).ValuesAsNumpy(),  # wind_speed_10m_max
                'Zip5Code': row['Zip5Code']
            }
            
            # Convert to DataFrame
            data = pd.DataFrame(daily_data)
            
            # Convert timestamp to date
            data['time'] = data['time'].dt.date
            
            # Convert units to match the Meteostat units (Celsius, mm, cm, km/h)
            # Open-Meteo returns temperatures in Celsius, precipitation in mm, snow in cm, wind in km/h by default
            
            records.append(data)
            
        except Exception as e:
            print(f"Error fetching data for zip code {row['Zip5Code']}: {e}")
            
            # Create a dataframe with NaN values for the date range
            date_range = pd.date_range(start=start, end=end)
            empty_data = pd.DataFrame({
                'time': date_range,
                'tavg': np.nan,
                'tmin': np.nan,
                'tmax': np.nan,
                'prcp': np.nan,
                'snow': np.nan,
                'wspd': np.nan,
                'Zip5Code': row['Zip5Code']
            })
            empty_data['time'] = empty_data['time'].dt.date
            records.append(empty_data)
    
    if records:
        # Create weather DataFrame at ZIP code + date level
        weather_df = pd.concat(records, ignore_index=True)
        
        # Now merge with original zip_df to get store-level data
        # This preserves all columns from the original zip_df
        result_df = pd.merge(
            zip_df,
            weather_df,
            on='Zip5Code',
            how='inner'
        )
        
        # Add load_time column
        result_df['load_time'] = datetime.now()
        return result_df
    else:
        # Return empty DataFrame with base structure
        empty_df = pd.DataFrame()
        empty_df['load_time'] = datetime.now()
        return empty_df

if REFETCH_BULK or not table_exists(BULK_DATA_TABLE):
    print(f"Fetching bulk weather data for all ZIP codes...")
    #bulk_df = fetch_bulk_weather(zip_df, years=BULK_YEARS) # for testing, using static zip code list
    bulk_df = fetch_bulk_weather(df_zip_store[:5], years=BULK_YEARS)
    
    # Convert pandas DataFrame to Spark DataFrame
    spark_bulk_df = spark.createDataFrame(bulk_df)
    
    # Write to Databricks table (overwrite if enabled)
    spark.sql(f"DROP TABLE IF EXISTS {BULK_DATA_TABLE}")
    spark_bulk_df.write.format("delta").mode("overwrite").saveAsTable(BULK_DATA_TABLE)
    print(f"Saved bulk data to table {BULK_DATA_TABLE}")
else:
    print(f"Loading cached bulk data from table {BULK_DATA_TABLE}")
    # Read from Databricks table
    spark_bulk_df = spark.table(BULK_DATA_TABLE)
    bulk_df = spark_bulk_df.toPandas()

display(bulk_df)

In [0]:

df_0 = bulk_df[bulk_df['StoreNumber'] == '10012']

display(df_0)

## Testing Code - Direct API Hit

In [0]:
def test_weather_api(latitude, longitude, start_date, end_date):
    """
    Test the Open-Meteo API for specific coordinates and date range.
    
    Args:
        latitude (float): Latitude coordinate
        longitude (float): Longitude coordinate
        start_date (str): Start date in 'YYYY-MM-DD' format
        end_date (str): End date in 'YYYY-MM-DD' format
        
    Returns:
        DataFrame: Weather data for the specified coordinates and date range with date as a column
    """
    # Setup Open-Meteo client
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    # Define API URL and parameters
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ["temperature_2m_mean", "temperature_2m_min", "temperature_2m_max", 
                  "precipitation_sum", "snowfall_sum", "wind_speed_10m_max"],
        "timezone": "auto"
    }
    
    print(f"Testing API for coordinates ({latitude}, {longitude}) from {start_date} to {end_date}")
    
    try:
        # Make API request
        responses = openmeteo.weather_api(url, params=params)
        response = responses[0]
        
        # Process daily data
        daily = response.Daily()
        
        # Process the Time() values which are returned as Unix timestamp strings
        daily_time_values = daily.Time()
        
        # Convert Unix timestamps (in seconds) to datetime objects
        daily_data = {
            'time': pd.to_datetime(daily_time_values, unit='s'),  # Convert Unix timestamps to datetime
            'tavg': daily.Variables(0).ValuesAsNumpy(),  # temperature_2m_mean
            'tmin': daily.Variables(1).ValuesAsNumpy(),  # temperature_2m_min
            'tmax': daily.Variables(2).ValuesAsNumpy(),  # temperature_2m_max
            'prcp': daily.Variables(3).ValuesAsNumpy(),  # precipitation_sum
            'snow': daily.Variables(4).ValuesAsNumpy(),  # snowfall_sum 
            'wspd': daily.Variables(5).ValuesAsNumpy(),  # wind_speed_10m_max
        }
        
        # Convert to DataFrame
        data = pd.DataFrame(daily_data)
        
        # Convert timestamp to date
        data['time'] = data['time'].dt.date
        
        if not data.empty:
            print(f"\nFound {len(data)} days of data")
            print("\nSummary statistics:")
            print(data.describe())
            return data
        else:
            print("No data found for the specified parameters")
            return pd.DataFrame(columns=['time'])
            
    except Exception as e:
        print(f"Error fetching data: {e}")
        return pd.DataFrame(columns=['time'])

# Example usage:
# test_data = test_weather_api(37.7749, -122.4194, '2023-01-01', '2023-01-15')  # San Francisco
# display(test_data)

# Test for a specific ZIP code's coordinates
suspect_zip = "54786"  # Replace with your suspicious ZIP code
test_zip_row = df_zip_store[df_zip_store['Zip5Code'] == suspect_zip].iloc[0]
lat, long = test_zip_row['latitude'], test_zip_row['longitude']

# Test for a specific date range where you observed weird values
suspect_data = test_weather_api(lat, long, '2024-01-01', '2024-05-31')
display(suspect_data)

In [0]:
unique_zips_df = df_zip_store.drop_duplicates(subset=['Zip5Code']).sort_values('Zip5Code').reset_index(drop=True)

display(unique_zips_df)

In [0]:
test_zip_row = df_zip_store[df_zip_store['Zip5Code'] == "54786"]
display(test_zip_row)

In [0]:
%sql 
-- -- summary stats for bulk fetch
-- Select  zip_code, 
--         count(*),
--         round(avg(tavg),2) as avg_temp,
--         round(avg(tmin),2) as avg_tmin,
--         round(avg(tmax),2) as avg_tmax
-- from cma_users_esrivast.weather_data_5_26_25
-- group by 1

## Incremental Fetch: Most Recent N Days

In [0]:
def fetch_recent_weather(zip_df, n_days=7):
    # Setup the Open-Meteo API client with cache and retry
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    # Define date range
    end = datetime.now().date() - timedelta(days=1)
    start = end - timedelta(days=n_days-1)
    
    # Format dates as strings for the API
    start_str = start.strftime("%Y-%m-%d")
    end_str = end.strftime("%Y-%m-%d")
    
    # Define the API URL
    url = "https://api.open-meteo.com/v1/forecast"
    
    records = []
    for _, row in zip_df.iterrows():
        lat, lon = row['latitude'], row['longitude']
        
        # Define parameters for this location
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_str,
            "end_date": end_str,
            "daily": ["temperature_2m_mean", "temperature_2m_min", "temperature_2m_max", 
                      "precipitation_sum", "snowfall_sum", "wind_speed_10m_max"],
            "timezone": "auto"
        }
        
        try:
            # Make the API request
            responses = openmeteo.weather_api(url, params=params)
            response = responses[0]
            
            # Process daily data
            daily = response.Daily()
            
            # Process the Time() values which are returned as Unix timestamp strings
            daily_time_values = daily.Time()
            
            # Convert Unix timestamps (in seconds) to datetime objects
            daily_data = {
                'time': pd.to_datetime(daily_time_values, unit='s'),  # Convert Unix timestamps to datetime
                'tavg': daily.Variables(0).ValuesAsNumpy(),  # temperature_2m_mean
                'tmin': daily.Variables(1).ValuesAsNumpy(),  # temperature_2m_min
                'tmax': daily.Variables(2).ValuesAsNumpy(),  # temperature_2m_max
                'prcp': daily.Variables(3).ValuesAsNumpy(),  # precipitation_sum
                'snow': daily.Variables(4).ValuesAsNumpy(),  # snowfall_sum 
                'wspd': daily.Variables(5).ValuesAsNumpy(),  # wind_speed_10m_max
                'zip_code': row['zip_code']
            }
            
            # Convert to DataFrame
            data = pd.DataFrame(daily_data)
            
            # Convert timestamp to date
            data['time'] = data['time'].dt.date
            
            # Ensure we have only the required columns in the right order
            data = data[['zip_code', 'time', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd']]
            records.append(data)
            
        except Exception as e:
            print(f"Error fetching recent data for zip code {row['zip_code']}: {e}")
    
    if records:
        result_df = pd.concat(records, ignore_index=True)
        # Add load_time column
        result_df['load_time'] = datetime.now()
        return result_df
    else:
        return pd.DataFrame()

recent_df = fetch_recent_weather(zip_df, n_days=N_DAYS)
display(recent_df)

In [0]:
# summary stats for most recent fetch
recent_df.groupby('zip_code').agg(
  count=('zip_code','count'),
  avg_temp=('tavg',lambda x: round(x.mean(),2)),
  avg_tmin=('tmax',lambda x: round(x.mean(),2)),
  avg_tmax=('tmin',lambda x: round(x.mean(),2))
).reset_index()

## Append New Data to Bulk Data Table

In [0]:
bulk_df.info()

In [0]:
recent_df.info()

In [0]:
def append_new_data(spark_bulk_df, new_df):
    # Convert the new DataFrame to a Spark DataFrame
    spark_new_df = spark.createDataFrame(new_df)
    
    # Create a temporary view for the new data
    spark_new_df.createOrReplaceTempView("new_data")
    
    # Check if any new records already exist in the bulk table
    # This is more efficient than a full join when we're just appending new records
    spark.sql(f"""
    MERGE INTO {BULK_DATA_TABLE} target
    USING new_data source
    ON target.zip_code = source.zip_code AND target.time = source.time
    WHEN MATCHED THEN
      UPDATE SET 
        tavg = source.tavg,
        tmin = source.tmin,
        tmax = source.tmax,
        prcp = source.prcp,
        snow = source.snow,
        wspd = source.wspd,
        load_time = source.load_time
    WHEN NOT MATCHED THEN
      INSERT *
    """)
    
    # Read the updated table
    updated_df = spark.table(BULK_DATA_TABLE).toPandas()
    return updated_df

# Check if bulk table exists and has data
if table_exists(BULK_DATA_TABLE):
    spark_bulk_df = spark.table(BULK_DATA_TABLE)
    full_df = append_new_data(spark_bulk_df, recent_df)
    print(f"Appended new data. Full dataset now has {len(full_df)} rows.")
else:
    # If bulk table doesn't exist, create it with the recent data
    spark_recent_df = spark.createDataFrame(recent_df)
    spark_recent_df.write.format("delta").mode("overwrite").saveAsTable(BULK_DATA_TABLE)
    full_df = recent_df
    print(f"Created new bulk table with {len(full_df)} rows.")

full_df.tail()

In [0]:
%sql 
-- summary stats for bulk fetch
Select  zip_code, 
        count(*),
        round(avg(tavg),2) as avg_temp,
        round(avg(tmin),2) as avg_tmin,
        round(avg(tmax),2) as avg_tmax
from cma_users_esrivast.weather_data_5_26_25
group by 1

## Data Quality Diagnostics

In [0]:
# Simple diagnostics
print("Missing values by column:")
print(full_df.isnull().sum())
print("\nSample data:")
print(full_df.head())

## Save Results (Optional Export to CSV)

In [0]:
# Optionally save the final results to CSV for external use
#full_df.to_csv("/dbfs/FileStore/weather_by_zip_final.csv", index=False)
#print("Saved final results to /dbfs/FileStore/weather_by_zip_final.csv")

# You can also query the table directly using Databricks SQL
#spark.sql(f"SELECT * FROM {BULK_DATA_TABLE} LIMIT 5").show()

# Optimize the Delta table (optional)
spark.sql(f"OPTIMIZE {BULK_DATA_TABLE}")
print(f"Optimized the {BULK_DATA_TABLE} Delta table")

## Table Summary Information

Databricks manages Spark sessions automatically, so we don't need to explicitly close them.

In [0]:
# No need to explicitly stop the Spark session in Databricks
# It will be managed by the Databricks environment

# Display summary information about the table
spark.sql(f"DESCRIBE EXTENDED {BULK_DATA_TABLE}").show(truncate=False)